# NYC Green Taxi — Ride Duration Prediction
**Baseline → Model comparison → Local MLflow tracking → Model Registry → ONNX export**

This notebook predicts the **duration (in minutes)** of NYC green-taxi trips (January 2023) and walks through the full MLOps flow:

| # | Section | What happens |
|---|---------|--------------|
| 1 | Setup | imports, paths, parameters, data download (only if the file is missing) |
| 2 | Load & explore | read the raw parquet file and inspect it |
| 3 | Clean the data | build the target, remove bad rows, remove duplicates |
| 4 | Features | keep only information known **before** the trip starts, save the clean file |
| 5 | Split | train / validation / test |
| 6 | Baseline | Random Forest baseline, evaluated on the validation set |
| 7 | MLflow (local) | connect to the local MLflow server and log the baseline run |
| 8 | Compare models | Linear Regression, Decision Tree, Random Forest, XGBoost |
| 9 | Register | register the selected model in the MLflow Model Registry |
| 10 | Load & predict | load the registered model and predict |
| 11 | Production | copy it to `NYC-prod` and give it the `champion` alias |
| 12 | ONNX | export the model to ONNX and verify it |

**Before running:** start the local MLflow server in a terminal opened in the project folder and leave it running:
`mlflow ui --backend-store-uri sqlite:///mlflow.db`

**Then run it top-to-bottom with *Run All*.** Requirements: `pandas pyarrow scikit-learn xgboost mlflow skops skl2onnx onnxruntime`.


## 1. Setup
### 1.1 Imports
All imports are in one place at the top.

In [1]:
# --- Standard library ---
import time
import urllib.request
from pathlib import Path

# --- Data & numerics ---
import numpy as np
import pandas as pd

# --- Scikit-learn ---
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    mean_squared_error,
    root_mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
    max_error,
)

# --- XGBoost ---
from xgboost import XGBRegressor

# --- Experiment tracking & model registry ---
import mlflow
import mlflow.sklearn
import mlflow.xgboost

# --- ONNX export ---
import onnxruntime as rt
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

### 1.2 Configuration
Paths and parameters live in one cell so every later cell uses the same values.

* `PARAMS` keeps the parameters you defined. The ones used in this baseline are `seed`, `min_duration_min`, `max_duration_min`, `val_size` and `test_size`. The distance, passenger and rare-category values are kept for later improvements but are **not applied yet**.
* `MONTH_START` / `MONTH_END` mark the month of the file (the raw data contains a few trips dated 2009 that must be removed).

In [2]:
SEED = 42

# Project folder and data paths
ROOT       = "/home/tamer/projects/mlops-practitioner"
RAW_DATA   = r"/home/tamer/projects/mlops-practitioner/data/raw/green_tripdata_2023-01.parquet"
CLEAN_FILE = r"/home/tamer/projects/mlops-practitioner/data/cleaned/green_tripdata_2023-01_clean.parquet"
ONNX_FILE  = f"{ROOT}/models/nyc_ride_duration.onnx"     # output of the ONNX export (section 12)

# Same file you downloaded with: wget <DATA_URL>
DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-01.parquet"

# Local MLflow server (started with: mlflow ui --backend-store-uri sqlite:///mlflow.db)
TRACKING_URI = "http://127.0.0.1:5000"

# The month covered by the file
MONTH_START = "2023-01-01"
MONTH_END   = "2023-02-01"   # exclusive

PARAMS = {
    "seed": SEED,
    "min_duration_min": 1,      # shorter trips are meter glitches
    "max_duration_min": 60,     # longer trips are very rare
    "min_distance_mi": 0.1,     # (not applied in this baseline)
    "max_distance_mi": 30,      # (not applied in this baseline)
    "max_passengers": 6,        # (not applied in this baseline)
    "val_size": 0.15,
    "test_size": 0.15,
    "rare_category_min_freq": 30,   # (not applied in this baseline)
}

### 1.3 Get the data
If the parquet file is not on disk yet, download it (this is the same file as the `wget` command). If it already exists, nothing happens.

In [3]:
raw_path = Path(RAW_DATA)
if not raw_path.exists():
    raw_path.parent.mkdir(parents=True, exist_ok=True)   # create data/raw if needed
    urllib.request.urlretrieve(DATA_URL, raw_path)
    print("Downloaded:", raw_path)
else:
    print("Data already available:", raw_path)

Data already available: /home/tamer/projects/mlops-practitioner/data/raw/green_tripdata_2023-01.parquet


## 2. Load & explore the raw data

In [4]:
df_raw = pd.read_parquet(RAW_DATA)
df_raw.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,1.0,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,0.5,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,0.5,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00


In [5]:
# Column types and missing values
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 68211 entries, 0 to 68210
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               68211 non-null  int64         
 1   lpep_pickup_datetime   68211 non-null  datetime64[us]
 2   lpep_dropoff_datetime  68211 non-null  datetime64[us]
 3   store_and_fwd_flag     63887 non-null  str           
 4   RatecodeID             63887 non-null  float64       
 5   PULocationID           68211 non-null  int64         
 6   DOLocationID           68211 non-null  int64         
 7   passenger_count        63887 non-null  float64       
 8   trip_distance          68211 non-null  float64       
 9   fare_amount            68211 non-null  float64       
 10  extra                  68211 non-null  float64       
 11  mta_tax                68211 non-null  float64       
 12  tip_amount             68211 non-null  float64       
 13  tolls_amount

In [6]:
# Summary statistics — note the pickup dates in 2009 and the extreme values
df_raw.describe()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
count,68211.000000,68211,68211,63887.000000,68211.000000,68211.000000,63887.000000,68211.000000,68211.000000,68211.000000,68211.000000,68211.000000,68211.000000,68211.000000,68211.000000,63887.000000,63877.000000,63887.000000
mean,1.863028,2023-01-16 20:10:55.679523,2023-01-16 20:29:01.515767,1.117160,98.549735,138.429901,1.315870,8.114852,16.603545,0.825431,0.588340,2.139012,0.169627,0.951625,21.789377,1.373613,1.021933,0.726748
min,1.000000,2009-01-01 20:21:27,2009-01-02 11:07:31,1.000000,1.000000,1.000000,0.000000,0.000000,-70.000000,-2.500000,-0.500000,-10.500000,0.000000,-1.000000,-71.500000,1.000000,1.000000,-2.750000
25%,2.000000,2023-01-09 11:59:47.500000,2023-01-09 12:16:37.500000,1.000000,74.000000,74.000000,1.000000,1.110000,9.300000,0.000000,0.500000,0.000000,0.000000,1.000000,12.900000,1.000000,1.000000,0.000000
50%,2.000000,2023-01-17 08:40:42,2023-01-17 08:56:38,1.000000,75.000000,138.000000,1.000000,1.850000,13.500000,0.000000,0.500000,1.600000,0.000000,1.000000,17.900000,1.000000,1.000000,0.000000
75%,2.000000,2023-01-24 15:52:30,2023-01-24 16:06:56,1.000000,129.000000,219.000000,1.000000,3.210000,19.800000,1.000000,0.500000,3.330000,0.000000,1.000000,26.150000,2.000000,1.000000,2.750000
max,2.000000,2023-02-01 03:10:05,2023-02-01 17:27:05,99.000000,265.000000,265.000000,9.000000,120098.840000,490.000000,12.500000,2.750000,222.220000,36.050000,1.000000,491.000000,5.000000,2.000000,2.750000
std,0.343820,NaN,NaN,1.372913,61.244314,76.761311,0.979054,585.105955,13.470121,1.269904,0.385819,3.052710,1.099789,0.196875,15.457115,0.508039,0.146465,1.212836


In [7]:
# Example: the duration of one trip is dropoff time - pickup time (a Timedelta)
df_raw["lpep_dropoff_datetime"][1] - df_raw["lpep_pickup_datetime"][1]

Timedelta('0 days 00:06:46')

## 3. Build the target and clean the data
### 3.1 Target: trip duration in minutes
`duration = dropoff time − pickup time`, converted from a Timedelta to minutes. We work on a copy (`df`) so `df_raw` stays untouched.

In [8]:
df = df_raw.copy()

# duration in minutes = (dropoff - pickup) in seconds / 60
df["duration"] = (df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]).dt.total_seconds() / 60
df["duration"].head()

0    11.016667
1     6.766667
2     6.333333
3     5.816667
4     5.966667
Name: duration, dtype: float64

### 3.2 Inspect the problems before filtering
Very long trips, zero-minute trips and pickups outside January 2023 would distort the model (and the error metrics), so we check how many there are.

In [9]:
print(df["duration"].describe(percentiles=[.01, .5, .99]).round(2).to_string())
print("\nnegative durations:", (df["duration"] < 0).sum())
print("shorter than min duration:", (df["duration"] < PARAMS["min_duration_min"]).sum())
print("longer than max duration:", (df["duration"] > PARAMS["max_duration_min"]).sum())
print("pickups outside the month:", (~df["lpep_pickup_datetime"].between(MONTH_START, MONTH_END, inclusive="left")).sum())
print("missing values per column with NaNs:")
print(df.isna().sum()[df.isna().sum() > 0].to_string())

count    68211.00
mean        18.10
std         74.93
min          0.00
1%           0.08
50%         11.40
99%         58.11
max       1439.88

negative durations: 0
shorter than min duration: 1645
longer than max duration: 620
pickups outside the month: 4
missing values per column with NaNs:
store_and_fwd_flag       4324
RatecodeID               4324
passenger_count          4324
ehail_fee               68211
payment_type             4324
trip_type                4334
congestion_surcharge     4324


### 3.3 Filter and de-duplicate
1. Keep realistic durations (`min_duration_min` … `max_duration_min` from `PARAMS`).
2. Keep only pickups inside the month of the file.
3. Remove exact duplicate rows. This is done **on the full rows** (before we drop any columns) so two different trips that only look alike after column removal are not deleted by mistake.

In [10]:
rows_before = len(df)

# 1. realistic trip durations
df = df[(df["duration"] >= PARAMS["min_duration_min"]) & (df["duration"] <= PARAMS["max_duration_min"])]

# 2. pickups inside January 2023 only
df = df[(df["lpep_pickup_datetime"] >= MONTH_START) & (df["lpep_pickup_datetime"] < MONTH_END)]

# 3. exact duplicate rows
df = df.drop_duplicates().copy()

print(f"rows: {rows_before:,} -> {len(df):,}  (removed {rows_before - len(df):,})")

rows: 68,211 -> 65,943  (removed 2,268)


## 4. Features
### 4.1 Choose the features
A ride-duration model must only use information that is known **before the trip starts**.

* **Kept:** pickup zone, dropoff zone, trip distance, pickup hour, pickup day-of-week.
* **Removed on purpose:** `fare_amount`, `total_amount`, `tip_amount`, `extra`, `mta_tax`, `tolls_amount`, `improvement_surcharge`. These are only known **after** the trip and the metered fare is calculated from time and distance, so they leak the answer into the model (this is *target leakage* and produces unrealistically perfect scores).
* **Removed:** the duplicate target column, IDs and columns that are mostly empty (`store_and_fwd_flag`, `RatecodeID`, `passenger_count`, `ehail_fee`, `payment_type`, `trip_type`, `congestion_surcharge`, `VendorID`).

In [11]:
# Time features from the pickup timestamp
df["pickup_hour"] = df["lpep_pickup_datetime"].dt.hour
df["pickup_dow"]  = df["lpep_pickup_datetime"].dt.dayofweek   # Monday = 0

FEATURES = ["PULocationID", "DOLocationID", "trip_distance", "pickup_hour", "pickup_dow"]
TARGET   = "duration"

# Drop any row that still has a missing value in the columns we use
df = df.dropna(subset=FEATURES + [TARGET])
df[FEATURES + [TARGET]].head()

,PULocationID,DOLocationID,trip_distance,pickup_hour,pickup_dow,duration
0,166,143,2.58,0,6,11.016667
1,24,43,1.81,0,6,6.766667
2,223,179,0.00,0,6,6.333333
3,41,238,1.30,0,6,5.816667
4,41,74,1.10,0,6,5.966667


### 4.2 Save the cleaned data and reload it
The cleaned file is written to `CLEAN_FILE` (the folder is created if needed) and read back so the rest of the notebook works from the saved file.

In [12]:
Path(CLEAN_FILE).parent.mkdir(parents=True, exist_ok=True)
df[FEATURES + [TARGET]].to_parquet(CLEAN_FILE, index=False)

df_model = pd.read_parquet(CLEAN_FILE)
print(df_model.shape)
df_model.head()

(65943, 6)


,PULocationID,DOLocationID,trip_distance,pickup_hour,pickup_dow,duration
0,166,143,2.58,0,6,11.016667
1,24,43,1.81,0,6,6.766667
2,223,179,0.00,0,6,6.333333
3,41,238,1.30,0,6,5.816667
4,41,74,1.10,0,6,5.966667


In [13]:
df_model.dtypes

PULocationID       int64
DOLocationID       int64
trip_distance    float64
pickup_hour        int32
pickup_dow         int32
duration         float64
dtype: object

In [14]:
# Features (X) and target (y)
X = df_model[FEATURES]
y = df_model[TARGET]
X.head()

,PULocationID,DOLocationID,trip_distance,pickup_hour,pickup_dow
0,166,143,2.58,0,6
1,24,43,1.81,0,6
2,223,179,0.00,0,6
3,41,238,1.30,0,6
4,41,74,1.10,0,6


## 5. Split the data
* **Train** — used to fit the models.
* **Validation** — used to compare models and choose one.
* **Test** — used once at the end to report the final score of the chosen model.

First 15 % is set aside for test, then 15 % of the *whole* data is taken from the rest for validation (`0.15 / 0.85` of what remains).

In [15]:
X_rest, X_test, y_rest, y_test = train_test_split(
    X, y, test_size=PARAMS["test_size"], random_state=SEED)

In [16]:
X_train, X_val, y_train, y_val = train_test_split(
    X_rest, y_rest,
    test_size=PARAMS["val_size"] / (1 - PARAMS["test_size"]),
    random_state=SEED)

print(f"train={len(X_train):,}  val={len(X_val):,}  test={len(X_test):,}")

train=46,159  val=9,892  test=9,892


> **Note on scaling:** the previous version scaled the features with `MinMaxScaler`. The models used here (linear regression on raw numbers, trees, forests, XGBoost) do not need scaling, and the scaler was never saved with the registered model, so the registered model would have received unscaled data in production. Removing it keeps training and prediction consistent.

## 6. Baseline: Random Forest
### 6.1 Train
A simple Random Forest that later models are compared against.

In [17]:
params = {
    "max_depth": 70,
    "random_state": 42,
    "n_estimators": 30,
    "n_jobs": -1,
    "verbose": 1,
    "criterion": 'absolute_error'
}

In [18]:
regr = RandomForestRegressor(**params)
regr.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:   12.4s finished


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",30
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'absolute_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",70
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",1
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, e

### 6.2 Evaluate on the validation set
One helper returns all six metrics so the same code is reused everywhere:
MSE, RMSE, MAE, R², MAPE and max error (RMSE and MAE are in **minutes**).

In [19]:
def regression_metrics(y_true, y_pred):
    """Return the six regression metrics used in this notebook as a dict."""
    return {
        "mse": mean_squared_error(y_true, y_pred),
        "rmse": root_mean_squared_error(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2_score": r2_score(y_true, y_pred),
        "mape": mean_absolute_percentage_error(y_true, y_pred),
        "max_error": max_error(y_true, y_pred),
    }

In [20]:
y_pred = regr.predict(X_val)
baseline_val_metrics = regression_metrics(y_val, y_pred)
pd.Series(baseline_val_metrics).round(4)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  30 out of  30 | elapsed:    0.2s finished


mse          23.7842
rmse          4.8769
mae           3.0629
r2_score      0.7137
mape          0.2640
max_error    47.1206
dtype: float64

## 7. Experiment tracking with MLflow (local server)
### 7.1 Connect to the local MLflow server
All runs are sent to the MLflow server on your machine. Watch them live in the UI at http://127.0.0.1:5000.

> **Start the server first** (once, in a terminal opened in the project folder, and leave it running):
> `mlflow ui --backend-store-uri sqlite:///mlflow.db`

The cell below checks that the server is reachable and stops with a clear message if it is not.

In [21]:
# Make sure the local MLflow server is running before we try to log anything
try:
    urllib.request.urlopen(f"{TRACKING_URI}/health", timeout=5)
except Exception as e:
    raise RuntimeError(
        f"MLflow server not reachable at {TRACKING_URI}.\n"
        "Start it in a terminal opened in the project folder:\n"
        "    mlflow ui --backend-store-uri sqlite:///mlflow.db"
    ) from e

mlflow.set_tracking_uri(TRACKING_URI)
print("Tracking to:", mlflow.get_tracking_uri())

Tracking to: http://127.0.0.1:5000


### 7.2 Log the baseline run
The already-trained baseline model (`regr`) is logged with its parameters and validation metrics — no need to train it a second time.
(`skops_trusted_types` tells MLflow it is safe to serialize the tree structure, because MLflow does not trust pickle files by default.)

In [22]:
mlflow.set_experiment("RandomForestRegressor Baseline")

with mlflow.start_run(run_name="RandomForest_Baseline"):
    # Parameters of the model we trained
    mlflow.log_params(params)

    # Metrics computed on the validation set
    mlflow.log_metrics({f"val_{name}": value for name, value in baseline_val_metrics.items()})

    # The trained model itself
    mlflow.sklearn.log_model(
        regr,
        name="model",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )

2026/09/30 08:39:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run RandomForest_Baseline at: http://127.0.0.1:5000/#/experiments/1/runs/82c9f7763f04428a8c6cbc997ce1e5bd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## 8. Compare multiple models
### 8.1 Models to try
Each entry is `(name, model class, parameters)`.

In [23]:
# Format: (Model Name, Model Class, Parameter Dictionary)
models_to_try = [
    (
        "Linear_Regression",
        LinearRegression,
        {"fit_intercept": True}
    ),
    (
        "Decision_Tree",
        DecisionTreeRegressor,
        {"max_depth": 100, "min_samples_split": 5, "random_state": 42, "criterion": "absolute_error"}
    ),
    (
        "Random_Forest",
        RandomForestRegressor,
        {"max_depth": 50, "n_estimators": 30, "min_samples_split": 5, "random_state": 42, "n_jobs": -1}
    ),
    (
        "XGBoost",
        XGBRegressor,
        {"max_depth": 20, "n_estimators": 30, "learning_rate": 0.1, "random_state": 42, "n_jobs": -1}
    )
]

### 8.2 Train, evaluate and log every model
For each model: train, measure the training time, score it on **validation** (used to choose) and **test** (for reporting only), log everything to MLflow, and remember the run/model information for the registry step.

In [24]:
mlflow.set_experiment("NYC Ride Duration")

results = {}          # per model: run id, model URI and metrics
trained_models = {}   # the fitted model objects (used later for checks and the ONNX export)

for model_name, ModelClass, model_params in models_to_try:
    with mlflow.start_run(run_name=model_name) as run:
        print(f"Training {model_name}...")

        # Log only the explicitly defined important parameters
        mlflow.log_params(model_params)

        # Initialize the model by unpacking the dictionary
        model = ModelClass(**model_params)

        # Track training time
        start_time = time.time()
        model.fit(X_train, y_train)
        training_time = time.time() - start_time

        # Predictions on validation (model choice) and test (final report)
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        val_rmse = root_mean_squared_error(y_val, val_pred)
        val_mae = mean_absolute_error(y_val, val_pred)
        test_rmse = root_mean_squared_error(y_test, test_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        mlflow.log_metrics({
            "val_rmse": val_rmse,
            "val_mae": val_mae,
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "training_time_seconds": training_time,
        })

        # Save the model artifact based on its specific library requirements
        if "XGBoost" in model_name:
            model_info = mlflow.xgboost.log_model(model, name="model")
        elif "Tree" in model_name or "Forest" in model_name:
            model_info = mlflow.sklearn.log_model(
                model,
                name="model",
                skops_trusted_types=["sklearn.tree._tree.Tree"],
            )
        else:
            model_info = mlflow.sklearn.log_model(model, name="model")

        results[model_name] = {
            "run_id": run.info.run_id,
            "model_uri": model_info.model_uri,   # use this URI to register the model
            "val_rmse": val_rmse, "val_mae": val_mae,
            "test_rmse": test_rmse, "test_mae": test_mae,
            "training_time_s": training_time,
        }
        trained_models[model_name] = model

        print(f"Finished {model_name} | val RMSE: {val_rmse:.4f} | val MAE: {val_mae:.4f}\n")

Training Linear_Regression...


2026/09/30 08:39:44 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Finished Linear_Regression | val RMSE: 9.0103 | val MAE: 6.6039

🏃 View run Linear_Regression at: http://127.0.0.1:5000/#/experiments/3/runs/5708e9cf732f483a9f95f603804445e4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Training Decision_Tree...


2026/09/30 08:39:55 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Finished Decision_Tree | val RMSE: 6.3258 | val MAE: 3.9027

🏃 View run Decision_Tree at: http://127.0.0.1:5000/#/experiments/3/runs/89b673508b4740fca351ec5b8a9deb54
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Training Random_Forest...


2026/09/30 08:40:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Finished Random_Forest | val RMSE: 4.8327 | val MAE: 3.0197

🏃 View run Random_Forest at: http://127.0.0.1:5000/#/experiments/3/runs/afa726bbcfb24debbc4b03d572332799
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Training XGBoost...


2026/09/30 08:40:33 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Finished XGBoost | val RMSE: 5.0050 | val MAE: 3.1049

🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/3/runs/58365ccfeb37462b9f29cdb73d6e65e5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


### 8.3 Compare the results
Sorted by validation RMSE (lower is better, in minutes).

In [25]:
leaderboard = (
    pd.DataFrame(results).T
    .drop(columns=["run_id", "model_uri"])
    .astype(float)
    .sort_values("val_rmse")
)
leaderboard.round(4)

,val_rmse,val_mae,test_rmse,test_mae,training_time_s
Random_Forest,4.8327,3.0197,4.8422,3.0211,1.9732
XGBoost,5.0050,3.1049,5.0397,3.1206,12.8538
Decision_Tree,6.3258,3.9027,6.5092,3.9942,1.7211
Linear_Regression,9.0103,6.6039,9.0213,6.5840,0.0593


### 8.4 Select the model
`Random_Forest` is used for registration and ONNX export. Change `SELECTED_MODEL` if another model wins in the table above.

> The ONNX export in section 12 supports scikit-learn models only (`Linear_Regression`, `Decision_Tree`, `Random_Forest`), not `XGBoost`.

In [26]:
SELECTED_MODEL = "Random_Forest"
print("Best model on validation :", leaderboard.index[0])
print("Selected model           :", SELECTED_MODEL)

Best model on validation : Random_Forest
Selected model           : Random_Forest


### 8.5 Final score on the test set
The test set is used only now, once, for the selected model.

In [27]:
y_pred = trained_models[SELECTED_MODEL].predict(X_test)
final_test_metrics = regression_metrics(y_test, y_pred)
pd.Series(final_test_metrics).round(4)

mse          23.4472
rmse          4.8422
mae           3.0211
r2_score      0.7175
mape          0.2617
max_error    48.2185
dtype: float64

## 9. Register the model
The model is registered from the URI MLflow returned when it was logged (`results[...]["model_uri"]`). This replaces the hard-coded run id and the `runs:/<run_id>/model` path, which MLflow 3 can no longer resolve.

In [28]:
model_name = SELECTED_MODEL
registered_model = mlflow.register_model(
    model_uri=results[model_name]["model_uri"],
    name=model_name,
)
model_version = registered_model.version
print(f"Success! Registered {model_name} version {model_version}")

Successfully registered model 'Random_Forest'.
2026/09/30 08:40:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Random_Forest, version 1


Success! Registered Random_Forest version 1


Created version '1' of model 'Random_Forest'.


## 10. Load the registered model and predict

In [29]:
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.pyfunc.load_model(model_uri)

In [30]:
y_pred = loaded_model.predict(X_test)
y_pred[:5]

array([10.24641601, 11.81812566, 11.06866931, 12.33624735, 14.47720072])

In [31]:
# Sanity check: the loaded model must give the same predictions as the model we trained
print("Same predictions as the trained model:", np.allclose(y_pred, trained_models[model_name].predict(X_test)))

Same predictions as the trained model: True


## 11. Promote the model to production
Instead of the old "stages", the registry uses **copies** and **aliases**:
1. Copy the model version into the production registered model `NYC-prod`.
2. Give that new version the alias `champion`, so serving code can always load `models:/NYC-prod@champion`.

In [32]:
current_model_uri = f"models:/{model_name}/{model_version}"
production_model_name = "NYC-prod"

client = mlflow.MlflowClient()
prod_version = client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

# Point the 'champion' alias at the version we just created
client.set_registered_model_alias(production_model_name, "champion", prod_version.version)
print(f"{production_model_name} version {prod_version.version} is now @champion")

Registered model 'NYC-prod' already exists. Creating a new version of this model...
Copied version '1' of model 'Random_Forest' to version '2' of model 'NYC-prod'.


NYC-prod version 2 is now @champion


In [33]:
prod_model_uri = f"models:/{production_model_name}@champion"

loaded_model = mlflow.pyfunc.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([10.24641601, 11.81812566, 11.06866931, 12.33624735])

## 12. Export the model to ONNX
ONNX is a portable format that can be served without scikit-learn (for example with `onnxruntime`).
### 12.1 Define the input
The input is a float tensor with a **dynamic batch size** (`None`) and one column per feature.

In [34]:
num_features = X_train.shape[1]
print("number of features:", num_features)

number of features: 5


### 12.2 Convert and save

In [35]:
initial_types = [("float_input", FloatTensorType([None, num_features]))]

# convert the scikit-learn model to ONNX
onnx_model = convert_sklearn(trained_models[SELECTED_MODEL], initial_types=initial_types)

# save it to disk (the folder is created if needed)
Path(ONNX_FILE).parent.mkdir(parents=True, exist_ok=True)
with open(ONNX_FILE, "wb") as f:
    f.write(onnx_model.SerializeToString())
print("Saved:", ONNX_FILE)

Saved: /home/tamer/projects/mlops-practitioner/models/nyc_ride_duration.onnx


### 12.3 Verify the ONNX model
Run the ONNX file with `onnxruntime` and compare with scikit-learn. ONNX works in 32-bit floats, so tiny differences are normal.

In [36]:
sess = rt.InferenceSession(ONNX_FILE, providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name

onnx_pred = sess.run(None, {input_name: X_test.to_numpy(dtype=np.float32)})[0].ravel()
sk_pred = trained_models[SELECTED_MODEL].predict(X_test)

diff = np.abs(onnx_pred - sk_pred)
print(f"mean abs difference: {diff.mean():.6f} min | max abs difference: {diff.max():.4f} min")
onnx_pred[:5]

mean abs difference: 0.000001 min | max abs difference: 0.0000 min


array([10.246416, 11.818127, 11.068669, 12.336247, 14.4772  ],
      dtype=float32)

---
### Done
The model was trained, tracked in the local MLflow server, registered (`Random_Forest`), promoted (`NYC-prod@champion`) and exported to ONNX.